# ETL para recomendador y análisis de negocio Olist

Este notebook transforma el dataset consolidado de Olist en tablas limpias para análisis de negocio y para alimentar un sistema de recomendación.

El objetivo no es solo limpiar datos por deporte, ese pasatiempo triste de la humanidad moderna, sino preparar información que ayude a responder preguntas como:

- ¿Qué productos conviene recomendar junto a los top de ventas?
- ¿Qué productos se venden más por región?
- ¿Qué temporadas tienen más y menos ventas?
- ¿Qué clientes y productos son mejores candidatos para campañas?
- ¿Qué datos faltan para responder cortes como género?
- ¿Qué tabla puede alimentar un recomendador basado en afinidad?

## Idea central

La propuesta busca recomendar productos relevantes o complementarios para impulsar conversión y ticket promedio. Por eso este ETL genera tablas de ventas, clientes, temporalidad y afinidad producto-producto / categoría-categoría.


In [1]:

# ============================================================
# ETL para sistema de recomendación y análisis de negocio Olist
# ============================================================

from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd


# -----------------------------
# Configuración
# -----------------------------

RAW_PATH = Path(r"C:\Users\Ismael2\Downloads\Brazilian E-Commerce Public Dataset by Olist.csv")
OUTPUT_DIR = Path("/mnt/data/olist_etl_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Si trabajas dentro del repositorio, puedes cambiar las rutas así:
# RAW_PATH = Path("../data/raw/Brazilian E-Commerce Public Dataset by Olist.csv")
# OUTPUT_DIR = Path("../data/processed")


# -----------------------------
# Extract
# -----------------------------

raw = pd.read_csv(RAW_PATH)
raw = raw.drop(columns=[c for c in ["Unnamed: 0"] if c in raw.columns])

print(f"Filas originales: {raw.shape[0]:,}")
print(f"Columnas originales: {raw.shape[1]:,}")


# -----------------------------
# Transformación base
# -----------------------------

date_cols = [
    "shipping_limit_date",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    raw[col] = pd.to_datetime(raw[col], errors="coerce")

raw["product_category_name"] = (
    raw["product_category_name"]
    .fillna("unknown")
    .astype(str)
    .str.strip()
    .str.lower()
)

for col in ["customer_city", "seller_city", "payment_type", "order_status"]:
    raw[col] = (
        raw[col]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .str.lower()
    )

for col in ["customer_state", "seller_state"]:
    raw[col] = (
        raw[col]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .str.upper()
    )

state_to_region = {
    "AC": "Norte",
    "AP": "Norte",
    "AM": "Norte",
    "PA": "Norte",
    "RO": "Norte",
    "RR": "Norte",
    "TO": "Norte",
    "AL": "Nordeste",
    "BA": "Nordeste",
    "CE": "Nordeste",
    "MA": "Nordeste",
    "PB": "Nordeste",
    "PE": "Nordeste",
    "PI": "Nordeste",
    "RN": "Nordeste",
    "SE": "Nordeste",
    "DF": "Centro-Oeste",
    "GO": "Centro-Oeste",
    "MT": "Centro-Oeste",
    "MS": "Centro-Oeste",
    "ES": "Sudeste",
    "MG": "Sudeste",
    "RJ": "Sudeste",
    "SP": "Sudeste",
    "PR": "Sul",
    "RS": "Sul",
    "SC": "Sul",
}

raw["customer_region"] = raw["customer_state"].map(state_to_region).fillna("unknown")
raw["seller_region"] = raw["seller_state"].map(state_to_region).fillna("unknown")


# -----------------------------
# Deduplicación analítica
# -----------------------------
# El dataset viene unido a nivel pedido-producto-pago.
# Si una orden tiene varios pagos, el mismo producto puede repetirse.
# Para ventas por producto se usa una fila única por order_id + order_item_id.
# Para ticket se agregan pagos únicos por order_id + payment_sequential.

raw_sorted = raw.sort_values(["order_id", "order_item_id", "payment_sequential"])
fact_order_items = raw_sorted.drop_duplicates(["order_id", "order_item_id"]).copy()

fact_order_items["item_key"] = (
    fact_order_items["order_id"].astype(str)
    + "-"
    + fact_order_items["order_item_id"].astype(str)
)

fact_order_items["product_volume_cm3"] = (
    fact_order_items["product_length_cm"]
    * fact_order_items["product_height_cm"]
    * fact_order_items["product_width_cm"]
)

fact_order_items["product_weight_kg"] = fact_order_items["product_weight_g"] / 1000
fact_order_items["line_total"] = (
    fact_order_items["price"].fillna(0)
    + fact_order_items["freight_value"].fillna(0)
)
fact_order_items["same_state"] = (
    fact_order_items["customer_state"] == fact_order_items["seller_state"]
).astype(int)

fact_order_items["purchase_date"] = fact_order_items["order_purchase_timestamp"].dt.date
fact_order_items["purchase_month"] = (
    fact_order_items["order_purchase_timestamp"].dt.to_period("M").astype(str)
)
fact_order_items["purchase_quarter"] = (
    fact_order_items["order_purchase_timestamp"].dt.to_period("Q").astype(str)
)
fact_order_items["purchase_year"] = fact_order_items["order_purchase_timestamp"].dt.year
fact_order_items["purchase_month_num"] = fact_order_items["order_purchase_timestamp"].dt.month
fact_order_items["purchase_dayofweek"] = (
    fact_order_items["order_purchase_timestamp"].dt.day_name()
)
fact_order_items["purchase_hour"] = fact_order_items["order_purchase_timestamp"].dt.hour

# El dataset público de Olist no incluye género.
# Se deja el campo como placeholder para una futura fuente externa.
fact_order_items["customer_gender"] = "unknown"


# -----------------------------
# Pagos y órdenes
# -----------------------------

payment_cols = [
    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_installments",
    "payment_value",
]

payments = raw[payment_cols].drop_duplicates().copy()

payments_agg = (
    payments
    .groupby("order_id")
    .agg(
        payment_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_methods_count=("payment_type", "nunique"),
    )
    .reset_index()
)

ptype = (
    payments
    .groupby(["order_id", "payment_type"], as_index=False)["payment_value"]
    .sum()
    .sort_values(["order_id", "payment_value"], ascending=[True, False])
)

main_payment_type = (
    ptype
    .drop_duplicates("order_id")[["order_id", "payment_type"]]
    .rename(columns={"payment_type": "main_payment_type"})
)

fact_orders = (
    fact_order_items
    .groupby("order_id")
    .agg(
        customer_id=("customer_id", "first"),
        customer_unique_id=("customer_unique_id", "first"),
        customer_city=("customer_city", "first"),
        customer_state=("customer_state", "first"),
        customer_region=("customer_region", "first"),
        order_status=("order_status", "first"),
        order_purchase_timestamp=("order_purchase_timestamp", "first"),
        order_approved_at=("order_approved_at", "first"),
        order_delivered_carrier_date=("order_delivered_carrier_date", "first"),
        order_delivered_customer_date=("order_delivered_customer_date", "first"),
        order_estimated_delivery_date=("order_estimated_delivery_date", "first"),
        n_items=("item_key", "nunique"),
        n_products=("product_id", "nunique"),
        n_categories=("product_category_name", "nunique"),
        n_sellers=("seller_id", "nunique"),
        item_revenue=("price", "sum"),
        freight_total=("freight_value", "sum"),
    )
    .reset_index()
    .merge(payments_agg, on="order_id", how="left")
    .merge(main_payment_type, on="order_id", how="left")
)

fact_orders["ticket_total"] = fact_orders["payment_total"].fillna(
    fact_orders["item_revenue"] + fact_orders["freight_total"]
)
fact_orders["item_ticket"] = fact_orders["item_revenue"]
fact_orders["freight_ratio"] = np.where(
    fact_orders["ticket_total"] > 0,
    fact_orders["freight_total"] / fact_orders["ticket_total"],
    np.nan,
)

fact_orders["purchase_date"] = fact_orders["order_purchase_timestamp"].dt.date
fact_orders["purchase_month"] = (
    fact_orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
)
fact_orders["purchase_quarter"] = (
    fact_orders["order_purchase_timestamp"].dt.to_period("Q").astype(str)
)
fact_orders["purchase_year"] = fact_orders["order_purchase_timestamp"].dt.year
fact_orders["purchase_month_num"] = fact_orders["order_purchase_timestamp"].dt.month
fact_orders["purchase_hour"] = fact_orders["order_purchase_timestamp"].dt.hour
fact_orders["purchase_dayofweek"] = fact_orders["order_purchase_timestamp"].dt.day_name()
fact_orders["customer_gender"] = "unknown"


def sales_season(month):
    """Etiqueta simple para exploración comercial. No reemplaza calendario promocional real."""
    if pd.isna(month):
        return "unknown"
    if month in [11, 12]:
        return "alta_fin_de_anio"
    if month in [1, 2]:
        return "baja_inicio_anio"
    if month in [5, 6, 7]:
        return "media_mitad_anio"
    return "regular"


fact_orders["sales_season"] = fact_orders["purchase_month_num"].apply(sales_season)
fact_order_items["sales_season"] = fact_order_items["purchase_month_num"].apply(sales_season)


# -----------------------------
# Dimensiones
# -----------------------------

dim_customers = (
    fact_order_items
    .sort_values("order_purchase_timestamp")
    .groupby("customer_unique_id")
    .agg(
        customer_city=("customer_city", "last"),
        customer_state=("customer_state", "last"),
        customer_region=("customer_region", "last"),
        first_purchase_at=("order_purchase_timestamp", "min"),
        last_purchase_at=("order_purchase_timestamp", "max"),
        n_orders=("order_id", "nunique"),
        n_products=("product_id", "nunique"),
        n_categories=("product_category_name", "nunique"),
    )
    .reset_index()
)

dim_customers["customer_gender"] = "unknown"
dim_customers["is_recurrent"] = (dim_customers["n_orders"] > 1).astype(int)

dim_products = (
    fact_order_items
    .groupby("product_id")
    .agg(
        product_category_name=("product_category_name", "first"),
        product_name_lenght=("product_name_lenght", "first"),
        product_description_lenght=("product_description_lenght", "first"),
        product_photos_qty=("product_photos_qty", "first"),
        product_weight_g=("product_weight_g", "first"),
        product_length_cm=("product_length_cm", "first"),
        product_height_cm=("product_height_cm", "first"),
        product_width_cm=("product_width_cm", "first"),
        product_volume_cm3=("product_volume_cm3", "first"),
        qty_sold=("item_key", "nunique"),
        orders_count=("order_id", "nunique"),
        customers_count=("customer_unique_id", "nunique"),
        revenue=("price", "sum"),
        avg_price=("price", "mean"),
    )
    .reset_index()
)

dim_sellers = (
    fact_order_items
    .groupby("seller_id")
    .agg(
        seller_city=("seller_city", "first"),
        seller_state=("seller_state", "first"),
        seller_region=("seller_region", "first"),
        n_orders=("order_id", "nunique"),
        n_products=("product_id", "nunique"),
        revenue=("price", "sum"),
    )
    .reset_index()
)

date_min = fact_orders["order_purchase_timestamp"].min().normalize()
date_max = fact_orders["order_purchase_timestamp"].max().normalize()

dim_dates = pd.DataFrame({"date": pd.date_range(date_min, date_max, freq="D")})
dim_dates["date_key"] = dim_dates["date"].dt.strftime("%Y%m%d").astype(int)
dim_dates["year"] = dim_dates["date"].dt.year
dim_dates["month"] = dim_dates["date"].dt.month
dim_dates["month_name"] = dim_dates["date"].dt.month_name()
dim_dates["quarter"] = dim_dates["date"].dt.quarter
dim_dates["weekofyear"] = dim_dates["date"].dt.isocalendar().week.astype(int)
dim_dates["dayofweek"] = dim_dates["date"].dt.day_name()
dim_dates["is_weekend"] = dim_dates["date"].dt.dayofweek.isin([5, 6]).astype(int)
dim_dates["sales_season"] = dim_dates["month"].apply(sales_season)


# -----------------------------
# Marts de negocio
# -----------------------------

mart_sales_by_product = (
    fact_order_items
    .groupby(["product_id", "product_category_name"])
    .agg(
        qty_sold=("item_key", "nunique"),
        orders_count=("order_id", "nunique"),
        customers_count=("customer_unique_id", "nunique"),
        revenue=("price", "sum"),
        avg_price=("price", "mean"),
        avg_freight=("freight_value", "mean"),
    )
    .reset_index()
    .sort_values(["qty_sold", "revenue"], ascending=False)
)

mart_sales_by_region_product = (
    fact_order_items
    .groupby(["customer_region", "customer_state", "product_id", "product_category_name"])
    .agg(
        qty_sold=("item_key", "nunique"),
        orders_count=("order_id", "nunique"),
        customers_count=("customer_unique_id", "nunique"),
        revenue=("price", "sum"),
        avg_price=("price", "mean"),
    )
    .reset_index()
    .sort_values(
        ["customer_region", "qty_sold", "revenue"],
        ascending=[True, False, False],
    )
)

mart_sales_by_gender_product = (
    fact_order_items
    .groupby(["customer_gender", "product_id", "product_category_name"])
    .agg(
        qty_sold=("item_key", "nunique"),
        orders_count=("order_id", "nunique"),
        customers_count=("customer_unique_id", "nunique"),
        revenue=("price", "sum"),
    )
    .reset_index()
    .sort_values(
        ["customer_gender", "qty_sold", "revenue"],
        ascending=[True, False, False],
    )
)

mart_sales_by_season = (
    fact_orders
    .groupby(["purchase_year", "purchase_month_num", "purchase_month", "sales_season"])
    .agg(
        n_orders=("order_id", "nunique"),
        customers_count=("customer_unique_id", "nunique"),
        ticket_total=("ticket_total", "sum"),
        avg_ticket=("ticket_total", "mean"),
        median_ticket=("ticket_total", "median"),
        avg_items_per_order=("n_items", "mean"),
    )
    .reset_index()
    .sort_values(["purchase_year", "purchase_month_num"])
)

snapshot_date = fact_orders["order_purchase_timestamp"].max()

mart_customer_features = (
    fact_orders
    .groupby("customer_unique_id")
    .agg(
        n_orders=("order_id", "nunique"),
        first_purchase_at=("order_purchase_timestamp", "min"),
        last_purchase_at=("order_purchase_timestamp", "max"),
        total_spent=("ticket_total", "sum"),
        avg_ticket=("ticket_total", "mean"),
        max_ticket=("ticket_total", "max"),
        avg_items=("n_items", "mean"),
        total_items=("n_items", "sum"),
        categories_count=("n_categories", "sum"),
    )
    .reset_index()
)

mart_customer_features["recency_days"] = (
    snapshot_date - mart_customer_features["last_purchase_at"]
).dt.days
mart_customer_features["is_recurrent"] = (
    mart_customer_features["n_orders"] > 1
).astype(int)


# -----------------------------
# Afinidad categoría-categoría
# -----------------------------

total_orders = fact_orders["order_id"].nunique()

order_categories = (
    fact_order_items
    .groupby("order_id")["product_category_name"]
    .apply(lambda x: sorted(set(x.dropna())))
    .reset_index(name="categories")
)

cat_counts = Counter()
cat_pair_counts = Counter()

for categories in order_categories["categories"]:
    for category in categories:
        cat_counts[category] += 1

    if len(categories) > 1:
        for category_a, category_b in combinations(categories, 2):
            cat_pair_counts[(category_a, category_b)] += 1

category_affinity_rows = []

for (category_a, category_b), co_orders in cat_pair_counts.items():
    anchor_orders = cat_counts[category_a]
    recommended_orders = cat_counts[category_b]

    support = co_orders / total_orders
    confidence_a_to_b = co_orders / anchor_orders if anchor_orders else 0
    confidence_b_to_a = co_orders / recommended_orders if recommended_orders else 0
    lift = (
        (co_orders * total_orders) / (anchor_orders * recommended_orders)
        if anchor_orders and recommended_orders
        else np.nan
    )

    category_affinity_rows.append(
        [
            category_a,
            category_b,
            co_orders,
            anchor_orders,
            recommended_orders,
            support,
            confidence_a_to_b,
            confidence_b_to_a,
            lift,
        ]
    )

mart_category_affinity = pd.DataFrame(
    category_affinity_rows,
    columns=[
        "anchor_category",
        "recommended_category",
        "co_orders",
        "anchor_orders",
        "recommended_orders",
        "support",
        "confidence_anchor_to_recommended",
        "confidence_recommended_to_anchor",
        "lift",
    ],
).sort_values(["lift", "co_orders"], ascending=False)


# -----------------------------
# Afinidad producto-producto
# -----------------------------
# Para evitar asociaciones ruidosas de productos casi nunca vendidos,
# se consideran productos con al menos 5 órdenes.

min_product_orders = 5

order_products = (
    fact_order_items
    .groupby("order_id")["product_id"]
    .apply(lambda x: sorted(set(x.dropna())))
    .reset_index(name="products")
)

product_counts = Counter()

for products in order_products["products"]:
    for product in products:
        product_counts[product] += 1

eligible_products = {
    product for product, count in product_counts.items()
    if count >= min_product_orders
}

product_pair_counts = Counter()

for products in order_products["products"]:
    products = [product for product in products if product in eligible_products]

    if len(products) > 1:
        for product_a, product_b in combinations(products, 2):
            product_pair_counts[(product_a, product_b)] += 1

product_affinity_rows = []

for (product_a, product_b), co_orders in product_pair_counts.items():
    anchor_orders = product_counts[product_a]
    recommended_orders = product_counts[product_b]

    support = co_orders / total_orders
    confidence_a_to_b = co_orders / anchor_orders if anchor_orders else 0
    confidence_b_to_a = co_orders / recommended_orders if recommended_orders else 0
    lift = (
        (co_orders * total_orders) / (anchor_orders * recommended_orders)
        if anchor_orders and recommended_orders
        else np.nan
    )

    product_affinity_rows.append(
        [
            product_a,
            product_b,
            co_orders,
            anchor_orders,
            recommended_orders,
            support,
            confidence_a_to_b,
            confidence_b_to_a,
            lift,
        ]
    )

mart_product_affinity = pd.DataFrame(
    product_affinity_rows,
    columns=[
        "anchor_product_id",
        "recommended_product_id",
        "co_orders",
        "anchor_orders",
        "recommended_orders",
        "support",
        "confidence_anchor_to_recommended",
        "confidence_recommended_to_anchor",
        "lift",
    ],
).sort_values(["lift", "co_orders"], ascending=False)


# -----------------------------
# Candidatos para recomendador
# -----------------------------
# Esta tabla ya es direccional:
# dado un producto ancla, devuelve productos recomendables ordenados por score.

directional_rows = []

for (product_a, product_b), co_orders in product_pair_counts.items():
    for anchor_product, recommended_product in [
        (product_a, product_b),
        (product_b, product_a),
    ]:
        confidence = (
            co_orders / product_counts[anchor_product]
            if product_counts[anchor_product]
            else 0
        )
        lift = (
            (co_orders * total_orders)
            / (product_counts[product_a] * product_counts[product_b])
            if product_counts[product_a] and product_counts[product_b]
            else np.nan
        )

        directional_rows.append(
            [
                anchor_product,
                recommended_product,
                co_orders,
                product_counts[anchor_product],
                product_counts[recommended_product],
                confidence,
                lift,
            ]
        )

mart_recommendation_candidates = pd.DataFrame(
    directional_rows,
    columns=[
        "anchor_product_id",
        "recommended_product_id",
        "co_orders",
        "anchor_orders",
        "recommended_orders",
        "confidence",
        "lift",
    ],
)

if not mart_recommendation_candidates.empty:
    anchor_meta = dim_products[
        ["product_id", "product_category_name"]
    ].rename(
        columns={
            "product_id": "anchor_product_id",
            "product_category_name": "anchor_category",
        }
    )

    recommended_meta = dim_products[
        ["product_id", "product_category_name", "qty_sold", "revenue"]
    ].rename(
        columns={
            "product_id": "recommended_product_id",
            "product_category_name": "recommended_category",
            "qty_sold": "recommended_qty_sold",
            "revenue": "recommended_revenue",
        }
    )

    popularity_max = max(dim_products["qty_sold"].max(), 1)

    mart_recommendation_candidates = (
        mart_recommendation_candidates
        .merge(anchor_meta, on="anchor_product_id", how="left")
        .merge(recommended_meta, on="recommended_product_id", how="left")
    )

    mart_recommendation_candidates["score_affinity"] = (
        mart_recommendation_candidates["confidence"].fillna(0) * 0.60
        + mart_recommendation_candidates["lift"]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(0)
            .clip(0, 10) / 10 * 0.30
        + np.log1p(
            mart_recommendation_candidates["recommended_qty_sold"].fillna(0)
        ) / np.log1p(popularity_max) * 0.10
    )

    mart_recommendation_candidates = (
        mart_recommendation_candidates
        .sort_values(
            ["anchor_product_id", "score_affinity"],
            ascending=[True, False],
        )
        .groupby("anchor_product_id")
        .head(20)
        .reset_index(drop=True)
    )


# -----------------------------
# Reporte de calidad
# -----------------------------

data_quality_report = pd.DataFrame(
    [
        {
            "table": "raw",
            "column": col,
            "rows": len(raw),
            "missing_count": raw[col].isna().sum(),
            "missing_pct": raw[col].isna().mean(),
            "unique_count": raw[col].nunique(dropna=True),
            "dtype": str(raw[col].dtype),
        }
        for col in raw.columns
    ]
)

validation_summary = pd.DataFrame(
    [
        {
            "check": "raw_rows",
            "value": len(raw),
            "comment": "Filas originales después de eliminar índice auxiliar.",
        },
        {
            "check": "fact_order_items_rows",
            "value": len(fact_order_items),
            "comment": "Filas únicas por order_id + order_item_id.",
        },
        {
            "check": "fact_orders_rows",
            "value": len(fact_orders),
            "comment": "Órdenes únicas.",
        },
        {
            "check": "duplicated_item_keys",
            "value": fact_order_items.duplicated(["order_id", "order_item_id"]).sum(),
            "comment": "Debe ser 0 para evitar inflar ventas por producto.",
        },
        {
            "check": "duplicated_order_ids",
            "value": fact_orders.duplicated("order_id").sum(),
            "comment": "Debe ser 0 para evitar inflar ticket promedio.",
        },
        {
            "check": "customer_gender_available",
            "value": 0,
            "comment": "El dataset no contiene género. Requiere fuente externa si el negocio necesita este corte.",
        },
    ]
)


# -----------------------------
# Load
# -----------------------------

tables = {
    "fact_order_items.csv": fact_order_items,
    "fact_orders.csv": fact_orders,
    "dim_customers.csv": dim_customers,
    "dim_products.csv": dim_products,
    "dim_sellers.csv": dim_sellers,
    "dim_dates.csv": dim_dates,
    "mart_sales_by_product.csv": mart_sales_by_product,
    "mart_sales_by_region_product.csv": mart_sales_by_region_product,
    "mart_sales_by_gender_product.csv": mart_sales_by_gender_product,
    "mart_sales_by_season.csv": mart_sales_by_season,
    "mart_customer_features.csv": mart_customer_features,
    "mart_category_affinity.csv": mart_category_affinity,
    "mart_product_affinity.csv": mart_product_affinity,
    "mart_recommendation_candidates.csv": mart_recommendation_candidates,
    "data_quality_report.csv": data_quality_report,
    "validation_summary.csv": validation_summary,
}

for file_name, table in tables.items():
    output_path = OUTPUT_DIR / file_name
    table.to_csv(output_path, index=False)

print("\nArchivos generados:")
for file_name in tables:
    print(f"- {OUTPUT_DIR / file_name}")

print("\nValidación:")
display(validation_summary)


Filas originales: 113,390
Columnas originales: 38

Archivos generados:
- \mnt\data\olist_etl_outputs\fact_order_items.csv
- \mnt\data\olist_etl_outputs\fact_orders.csv
- \mnt\data\olist_etl_outputs\dim_customers.csv
- \mnt\data\olist_etl_outputs\dim_products.csv
- \mnt\data\olist_etl_outputs\dim_sellers.csv
- \mnt\data\olist_etl_outputs\dim_dates.csv
- \mnt\data\olist_etl_outputs\mart_sales_by_product.csv
- \mnt\data\olist_etl_outputs\mart_sales_by_region_product.csv
- \mnt\data\olist_etl_outputs\mart_sales_by_gender_product.csv
- \mnt\data\olist_etl_outputs\mart_sales_by_season.csv
- \mnt\data\olist_etl_outputs\mart_customer_features.csv
- \mnt\data\olist_etl_outputs\mart_category_affinity.csv
- \mnt\data\olist_etl_outputs\mart_product_affinity.csv
- \mnt\data\olist_etl_outputs\mart_recommendation_candidates.csv
- \mnt\data\olist_etl_outputs\data_quality_report.csv
- \mnt\data\olist_etl_outputs\validation_summary.csv

Validación:


,check,value,comment
0,raw_rows,113390,Filas originales después de eliminar índice au...
1,fact_order_items_rows,108640,Filas únicas por order_id + order_item_id.
2,fact_orders_rows,95128,Órdenes únicas.
3,duplicated_item_keys,0,Debe ser 0 para evitar inflar ventas por produ...
4,duplicated_order_ids,0,Debe ser 0 para evitar inflar ticket promedio.
5,customer_gender_available,0,El dataset no contiene género. Requiere fuente...


## Modelo analítico generado

El ETL genera una estructura cercana a un modelo estrella:

### Tablas base

- `fact_orders`: una fila por orden. Sirve para ticket promedio, temporalidad, métodos de pago y análisis de clientes.
- `fact_order_items`: una fila por producto dentro de una orden. Sirve para producto, categoría, seller y región.
- `dim_customers`: clientes únicos, ubicación, recurrencia y frecuencia.
- `dim_products`: productos únicos, categoría, dimensiones, fotos, peso y métricas de venta.
- `dim_sellers`: vendedores únicos.
- `dim_dates`: calendario analítico.

### Marts derivados

- `mart_sales_by_product`
- `mart_sales_by_region_product`
- `mart_sales_by_gender_product`
- `mart_sales_by_season`
- `mart_customer_features`
- `mart_category_affinity`
- `mart_product_affinity`
- `mart_recommendation_candidates`

La tabla más útil para el MVP de recomendación es `mart_recommendation_candidates`, porque recibe un `anchor_product_id` y devuelve productos candidatos con `confidence`, `lift` y `score_affinity`.


In [2]:

# ============================================================
# Ejemplos de preguntas de negocio que ya puede responder el ETL
# ============================================================

# 1. ¿Cuáles son los productos más afines a los top de ventas?
top_products = mart_sales_by_product.head(10)["product_id"].tolist()

affinity_for_top_products = (
    mart_recommendation_candidates[
        mart_recommendation_candidates["anchor_product_id"].isin(top_products)
    ]
    .sort_values(["anchor_product_id", "score_affinity"], ascending=[True, False])
    .groupby("anchor_product_id")
    .head(5)
)

display(affinity_for_top_products)


# 2. ¿Cuáles son los productos más vendidos en cada región?
top_products_by_region = (
    mart_sales_by_region_product
    .sort_values(["customer_region", "qty_sold", "revenue"], ascending=[True, False, False])
    .groupby("customer_region")
    .head(10)
)

display(top_products_by_region)


# 3. ¿Cuáles son los productos más populares entre hombres y mujeres?
# Nota honesta, porque parece que la realidad insiste en no traer todas las columnas:
# el dataset de Olist no contiene género, por lo que esta tabla queda en "unknown".
# Para responder de verdad, hay que enriquecer dim_customers con una fuente externa o dato de perfil.

top_products_by_gender = (
    mart_sales_by_gender_product
    .sort_values(["customer_gender", "qty_sold", "revenue"], ascending=[True, False, False])
    .groupby("customer_gender")
    .head(10)
)

display(top_products_by_gender)


# 4. ¿Cuáles son las temporadas de mayor y menor venta?
seasonality_summary = (
    mart_sales_by_season
    .sort_values("ticket_total", ascending=False)
)

display(seasonality_summary)


# 5. ¿Qué categorías conviene recomendar juntas?
top_category_affinity = (
    mart_category_affinity
    .query("co_orders >= 3")
    .sort_values(["lift", "co_orders"], ascending=False)
    .head(20)
)

display(top_category_affinity)


# 6. ¿Qué clientes son mejores candidatos para campañas de recompra?
recurrent_or_high_value_customers = (
    mart_customer_features
    .sort_values(["is_recurrent", "total_spent", "avg_ticket"], ascending=False)
    .head(20)
)

display(recurrent_or_high_value_customers)


# 7. ¿Qué productos tienen alto volumen y ticket atractivo?
high_value_products = (
    mart_sales_by_product
    .assign(score_business=lambda df: np.log1p(df["qty_sold"]) * df["avg_price"])
    .sort_values("score_business", ascending=False)
    .head(20)
)

display(high_value_products)


,anchor_product_id,recommended_product_id,co_orders,anchor_orders,recommended_orders,confidence,lift,anchor_category,recommended_category,recommended_qty_sold,recommended_revenue,score_affinity
172,154e7e31ebfa092203795c972e5804a6,7c1bd920dbdf22470b68bde975dd3ccf,3,262,214,0.011450,5.089962,beleza_saude,beleza_saude,220,13187.80,0.245860
173,154e7e31ebfa092203795c972e5804a6,2b4609f8948be18874494203496bc318,1,262,254,0.003817,1.429464,beleza_saude,beleza_saude,255,22277.27,0.133815
412,368c6c730842d78016ad823897a372db,53759a2ecddad2bb87a079a1f1519f73,8,291,287,0.027491,9.112205,ferramentas_jardim,ferramentas_jardim,373,20387.20,0.384562
413,368c6c730842d78016ad823897a372db,0bcc3eeca39e1064258aa1e932269894,4,291,100,0.013746,13.076014,ferramentas_jardim,ferramentas_jardim,115,6220.90,0.384235
414,368c6c730842d78016ad823897a372db,09b0d15a8cc9a84e7af7e0225f67dc45,1,291,14,0.003436,23.350025,ferramentas_jardim,moveis_decoracao,21,1407.00,0.351473
415,368c6c730842d78016ad823897a372db,a39cc58c1b5926b6f9f378daa89f1315,1,291,35,0.003436,9.340010,ferramentas_jardim,casa_construcao,35,5259.80,0.339546
416,368c6c730842d78016ad823897a372db,a5341e3f8155dbb3e62323d3ea289729,1,291,41,0.003436,7.973179,ferramentas_jardim,moveis_decoracao,41,3646.50,0.301005
443,389d119b48cf3043d311335e499d9c6b,422879e10f46682990de24d770e7f83d,11,309,352,0.035599,9.620550,ferramentas_jardim,ferramentas_jardim,484,26577.22,0.408831
444,389d119b48cf3043d311335e499d9c6b,53759a2ecddad2bb87a079a1f1519f73,9,309,287,0.029126,9.654071,ferramentas_jardim,ferramentas_jardim,373,20387.20,0.401799
445,389d119b48cf3043d311335e499d9c6b,0bcc3eeca39e1064258aa1e932269894,3,309,100,0.009709,9.235728,ferramentas_jardim,ferramentas_jardim,115,6220.90,0.358885


,customer_region,customer_state,product_id,product_category_name,qty_sold,orders_count,customers_count,revenue,avg_price
2656,Centro-Oeste,GO,9571759451b1d780ee7c15012ea109d4,automotivo,20,1,1,1974.00,98.700000
2806,Centro-Oeste,GO,aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,18,15,15,1278.60,71.033333
2071,Centro-Oeste,GO,368c6c730842d78016ad823897a372db,ferramentas_jardim,17,11,11,929.40,54.670588
531,Centro-Oeste,DF,4c2394abfbac7ff59ec7a420918562fa,beleza_saude,16,14,14,1359.84,84.990000
2078,Centro-Oeste,GO,37eb69aca8718e843d897aa7b82f462d,ferramentas_jardim,15,1,1,765.00,51.000000
2254,Centro-Oeste,GO,53b36df67ebb7c41585e8d54d6772e08,relogios_presentes,13,13,13,1478.49,113.730000
2685,Centro-Oeste,GO,99a4788cb24856965c36a24e339b6058,cama_mesa_banho,13,13,13,1162.70,89.438462
449,Centro-Oeste,DF,3fbc0ef745950c7932d5f2a446189725,beleza_saude,13,12,12,854.87,65.759231
1897,Centro-Oeste,GO,19c91ef95d509ea33eda93495c4d3481,beleza_saude,12,11,11,1440.88,120.073333
1185,Centro-Oeste,DF,aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,12,11,11,889.00,74.083333


,customer_gender,product_id,product_category_name,qty_sold,orders_count,customers_count,revenue
21240,unknown,aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,520,425,424,37104.30
8277,unknown,422879e10f46682990de24d770e7f83d,ferramentas_jardim,484,352,349,26577.22
18950,unknown,99a4788cb24856965c36a24e339b6058,cama_mesa_banho,477,456,456,42049.66
7076,unknown,389d119b48cf3043d311335e499d9c6b,ferramentas_jardim,390,309,308,21336.79
6800,unknown,368c6c730842d78016ad823897a372db,ferramentas_jardim,388,291,289,21056.80
10391,unknown,53759a2ecddad2bb87a079a1f1519f73,ferramentas_jardim,373,287,283,20387.20
25959,unknown,d1c427060a0f73f6b889a5c7c61f2ac4,informatica_acessorios,332,313,312,45620.56
10418,unknown,53b36df67ebb7c41585e8d54d6772e08,relogios_presentes,321,304,301,37454.63
2685,unknown,154e7e31ebfa092203795c972e5804a6,beleza_saude,274,262,261,6173.26
7739,unknown,3dd2a17168ec895c781a9191c1e95ad7,informatica_acessorios,272,253,252,40782.80


,purchase_year,purchase_month_num,purchase_month,sales_season,n_orders,customers_count,ticket_total,avg_ticket,median_ticket,avg_items_per_order
12,2017,11,2017-11,alta_fin_de_anio,7186,7082,1138311.73,158.406865,104.095,1.162817
18,2018,5,2018-05,media_mitad_anio,6716,6660,1125170.38,167.535792,108.615,1.156641
17,2018,4,2018-04,regular,6731,6677,1124085.95,167.001330,108.680,1.151835
16,2018,3,2018-03,regular,6884,6799,1101878.43,160.063688,105.640,1.145264
14,2018,1,2018-01,baja_inicio_anio,6901,6808,1054990.30,152.874989,103.370,1.136647
20,2018,7,2018-07,media_mitad_anio,6118,6059,1022259.00,167.090389,108.920,1.129944
19,2018,6,2018-06,media_mitad_anio,6075,6037,1008817.01,166.060413,112.090,1.149630
21,2018,8,2018-08,regular,6324,6283,982834.99,155.413503,100.955,1.124447
15,2018,2,2018-02,baja_inicio_anio,6451,6299,951941.42,147.564939,102.030,1.147419
13,2017,12,2017-12,alta_fin_de_anio,5383,5322,825902.84,153.427984,105.060,1.123909


,anchor_category,recommended_category,co_orders,anchor_orders,recommended_orders,support,confidence_anchor_to_recommended,confidence_recommended_to_anchor,lift
80,cama_mesa_banho,casa_conforto,43,9271,392,0.000452,0.004638,0.109694,1.125548
13,construcao_ferramentas_iluminacao,moveis_decoracao,11,242,6303,0.000116,0.045455,0.001745,0.686023
79,moveis_escritorio,moveis_sala,3,1254,414,0.000032,0.002392,0.007246,0.549708
21,casa_construcao,moveis_decoracao,13,483,6303,0.000137,0.026915,0.002063,0.406216
120,casa_construcao,ferramentas_jardim,7,483,3447,0.000074,0.014493,0.002031,0.399961
165,artes,moveis_decoracao,5,195,6303,0.000053,0.025641,0.000793,0.386987
20,audio,relogios_presentes,6,348,5493,0.000063,0.017241,0.001092,0.298587
92,moveis_decoracao,moveis_sala,7,6303,414,0.000074,0.001111,0.016908,0.255187
23,casa_conforto,moveis_decoracao,6,392,6303,0.000063,0.015306,0.000952,0.231008
15,malas_acessorios,papelaria,5,1019,2264,0.000053,0.004907,0.002208,0.206171


,customer_unique_id,n_orders,first_purchase_at,last_purchase_at,total_spent,avg_ticket,max_ticket,avg_items,total_items,categories_count,recency_days,is_recurrent
78557,da122df9eeddfedc1dc1f5349a1a690c,2,2017-04-01 15:58:40,2017-04-01 15:58:41,7571.63,3785.815000,4950.34,1.000000,2,2,514,1
72146,c8460e4251689ba205045f3ea17884a1,4,2018-08-07 09:03:02,2018-08-08 14:27:15,4655.91,1163.977500,1202.64,6.000000,24,4,21,1
32450,59d66d72939bc9497e19d89c61a96d5f,2,2017-03-02 12:13:18,2017-08-10 22:09:50,3559.99,1779.995000,2626.07,1.000000,2,2,383,1
84601,eae0a83d752b1dd32697e0e7b4221656,2,2018-02-01 18:32:02,2018-04-24 17:06:54,2783.01,1391.505000,1988.55,7.500000,15,2,126,1
77629,d77aa95864ae5b42160937615628723a,2,2017-08-29 22:35:42,2017-08-29 22:38:40,2450.10,1225.050000,1225.05,1.000000,2,2,364,1
48538,86df00dc5fd68f4dd5d5945ca19f3ed6,3,2017-06-08 10:49:42,2017-10-13 16:10:18,2400.48,800.160000,864.60,4.000000,12,3,319,1
44340,7b0eaf68a16e4808e5388c67345033c9,2,2018-05-19 11:16:55,2018-05-19 11:30:35,2340.08,1170.040000,1891.07,1.000000,2,2,102,1
39626,6ddbc64bd04d40f7768ff088d94cbeb8,2,2018-04-09 22:58:05,2018-04-09 22:58:05,2299.66,1149.830000,1149.83,1.000000,2,2,141,1
10766,1da09dd64e235e7c2f29a4faff33535c,3,2017-05-10 14:04:15,2018-01-11 11:16:49,2164.40,721.466667,859.36,3.333333,10,3,230,1
41597,73601b1eec55943e90ce8d61253d5c09,2,2017-09-26 17:29:19,2017-11-01 13:39:49,2068.68,1034.340000,1034.34,1.000000,2,2,301,1


,product_id,product_category_name,qty_sold,orders_count,customers_count,revenue,avg_price,avg_freight,score_business
8871,470433f95ba906e17efac3fce39e9ffd,informatica_acessorios,5,5,5,14999.45,2999.890000,40.222000,5375.081314
2891,16c4e87b98a9370a9cbc3a4658a3f45b,instrumentos_musicais,13,13,13,25034.00,1925.692308,122.780769,5082.012399
26515,d6160fb7873f184099d9bc95e30376af,pcs,33,33,32,45949.35,1392.404545,41.358485,4910.120423
31242,fd0065af7f09af4b82a0ca8f3eed1852,automotivo,10,10,10,19999.90,1999.990000,34.144000,4795.766567
9054,489ae2aa008f021502940f251d4cce7f,utilidades_domesticas,1,1,1,6735.00,6735.000000,194.310000,4668.346261
13137,69c590f7ffc7bf8db97190b6cb6ed62e,pcs,1,1,1,6729.00,6729.000000,193.210000,4664.187378
3550,1bdf5e6731585cf01aa8169c7028d6ad,artes,1,1,1,6499.00,6499.000000,227.660000,4504.763526
16838,87feb07adc221a4c6cdf051ea1afd0ff,eletrodomesticos_2,7,7,7,14770.00,2110.000000,52.587143,4387.621653
6629,34f99d82cfc355d08d8db780d14aa002,pcs,3,3,3,9399.97,3133.323333,126.496667,4343.708469
11016,588531f8ec37e7d5ff5b7b22ea0488f8,pcs,20,19,19,28291.99,1414.599500,39.411000,4306.779918


## Lectura para el proyecto

1. **Dashboard / análisis de negocio**  
   Permite visualizar ticket promedio, ventas por región, temporalidad, categorías top, productos top y comportamiento por cliente.

2. **Modelo de recomendación**  
   Permite construir un baseline de popularidad, recomendaciones por producto ancla y recomendaciones por categorías afines.

### Limitaciones reales

- El dataset no tiene información de género, por lo que no se debe afirmar popularidad entre hombres y mujeres sin enriquecer datos.
- La tasa de conversión real tampoco se calcula directamente porque faltan visitas, clics, impresiones, productos mostrados, carritos abandonados y sesiones.
- Lo que sí se puede construir es una evaluación offline con compras futuras o un proxy de afinidad a partir del historial de pedidos.
